# 04 — XGBoost Model Trained on Observed Flux

Train a model to predict observed plastic flux from Meijer Table S3.

### Key findings from exploration:
- 66 calibration rivers → 64 unique outfalls after dedup
- Meijer's R²≈0.71 was on **monthly** obs; annualized R² is only ~0.50
- XGBoost with 64 samples and 35 features: R²=0.16 (severe overfitting)
- Reducing features to 5-7: Ridge R²≈0.15-0.18 (features alone weak)
- Best approach: **linear correction of Meijer prediction** → R²=0.57 (LOO)
- Adding residual features **hurts** (overfitting noise with n=64)

### Strategy:
1. Reimplement Meijer's probabilistic model with updated inputs (WaW3.0, ERA5)
2. Apply linear calibration correction from observed flux
3. For rivers without waste data, use Meijer prediction directly

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

sys.path.insert(0, str(Path.cwd().parent / "src"))

DATA_PROC = Path("../data/processed")
DATA_RAW = Path("../data/raw")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

plt.rcParams['figure.dpi'] = 120

## 1. Load data

In [ ]:
obs = pd.read_csv(DATA_PROC / "observed_flux_matched_S3.csv")
fm = pd.read_csv(DATA_PROC / "feature_matrix_v1.csv")

# Dedup: keep river with highest observed flux per outfall
obs_dedup = obs.sort_values('obs_annual', ascending=False).drop_duplicates(subset='outfall_idx', keep='first')

print(f"Observed rivers: {len(obs)} → {len(obs_dedup)} after dedup")
print(f"Removed: {set(obs.name) - set(obs_dedup.name)}")
print(f"Feature matrix: {fm.shape}")

## 2. Establish baselines

In [ ]:
y = obs_dedup['log_obs'].values
log_me = np.log10(fm['emission_ton'].iloc[obs_dedup['outfall_idx'].values].clip(lower=0.01).values)

# Baseline 1: Meijer prediction alone
r2_me = r2_score(y, log_me)
sp_me = stats.spearmanr(y, log_me).statistic
print(f"Meijer prediction (no correction): R²={r2_me:.3f}, ρ={sp_me:.3f}")

# Baseline 2: Linear correction of Meijer (obs = a*meijer + b)
lr = LinearRegression()
y_pred_lr = cross_val_predict(lr, log_me.reshape(-1,1), y, cv=LeaveOneOut())
r2_lr = r2_score(y, y_pred_lr)
sp_lr = stats.spearmanr(y, y_pred_lr).statistic
rmse_lr = np.sqrt(mean_squared_error(y, y_pred_lr))
print(f"Linear correction of Meijer (LOO): R²={r2_lr:.3f}, ρ={sp_lr:.3f}, RMSE={rmse_lr:.3f}")

# Baseline 3: Ridge with features only (no Meijer)
feat_only = ['log_discharge', 'ORD_STRA', 'log_upstream_area', 'precip_cv', 'nightlight_intensity']
X_feat = fm[feat_only].iloc[obs_dedup['outfall_idx'].values].fillna(-999)
scaler = StandardScaler()
ridge = Ridge(alpha=5.0)
y_pred_ridge = cross_val_predict(ridge, scaler.fit_transform(X_feat), y, cv=LeaveOneOut())
r2_ridge = r2_score(y, y_pred_ridge)
sp_ridge = stats.spearmanr(y, y_pred_ridge).statistic
print(f"Ridge (5 features, no Meijer, LOO): R²={r2_ridge:.3f}, ρ={sp_ridge:.3f}")

print(f"\n→ Best baseline: Linear correction of Meijer (R²={r2_lr:.3f})")

## 3. Feature-only XGBoost (for comparison)

Test whether XGBoost with our updated features can beat the Meijer baseline.

In [ ]:
# Feature-only XGBoost with very conservative hyperparams (small n)
feature_cols = ['log_discharge', 'log_upstream_area', 'ORD_STRA', 'precip_cv',
                'nightlight_intensity', 'log_mpw_rate', 'runoff_mean_mm_yr']

X_xgb = fm[feature_cols].iloc[obs_dedup['outfall_idx'].values].fillna(-999)

xgb_model = xgb.XGBRegressor(
    n_estimators=50, max_depth=2, learning_rate=0.05,
    min_child_weight=5, reg_alpha=10, reg_lambda=10,
    subsample=0.8, colsample_bytree=0.8, random_state=42, missing=-999
)

y_pred_xgb = cross_val_predict(xgb_model, X_xgb, y, cv=LeaveOneOut())
r2_xgb = r2_score(y, y_pred_xgb)
sp_xgb = stats.spearmanr(y, y_pred_xgb).statistic
print(f"XGBoost (7 features, LOO): R²={r2_xgb:.3f}, ρ={sp_xgb:.3f}")
print(f"→ XGBoost alone underperforms Meijer correction with n=64")

## 4. Final model: Meijer + Linear Calibration

Given the small sample size (n=64), the best approach is:
1. Start with Meijer's probabilistic model predictions
2. Apply a linear calibration correction fit on observed flux
3. Our value-add comes from **updated input data** (WaW3.0, ERA5-Land, VIIRS)

In [ ]:
# Fit linear calibration on ALL observed rivers
lr_final = LinearRegression()
lr_final.fit(log_me.reshape(-1,1), y)

print(f"Calibration: log10(obs) = {lr_final.coef_[0]:.3f} × log10(Meijer) + {lr_final.intercept_:.3f}")
print(f"Slope < 1 means Meijer overestimates high-emission rivers")
print(f"Intercept > 0 means Meijer underestimates low-emission rivers")

# LOO evaluation of final model
y_pred_final = cross_val_predict(LinearRegression(), log_me.reshape(-1,1), y, cv=LeaveOneOut())
r2_final = r2_score(y, y_pred_final)
sp_final = stats.spearmanr(y, y_pred_final).statistic
rmse_final = np.sqrt(mean_squared_error(y, y_pred_final))

# Without Kuantan
no_k = obs_dedup['name'].values != 'Kuantan'
r2_nk = r2_score(y[no_k], y_pred_final[no_k])
sp_nk = stats.spearmanr(y[no_k], y_pred_final[no_k]).statistic

print(f"\nFinal model (LOO-CV):")
print(f"  All (n={len(y)}): R²={r2_final:.3f}, ρ={sp_final:.3f}, RMSE={rmse_final:.3f}")
print(f"  No Kuantan (n={no_k.sum()}): R²={r2_nk:.3f}, ρ={sp_nk:.3f}")
print(f"  Meijer monthly: R²≈0.71 (n=52)")

In [ ]:
# Observed vs Predicted plot
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y, y_pred_final, alpha=0.7, edgecolors='k', linewidth=0.5, zorder=3)
ax.scatter(log_me, y, alpha=0.3, c='red', marker='x', label='Meijer uncorrected', zorder=2)
lims = [min(y.min(), y_pred_final.min()) - 0.3, max(y.max(), y_pred_final.max()) + 0.3]
ax.plot(lims, lims, 'k--', lw=1, alpha=0.5, label='1:1')
ax.set_xlabel('log₁₀ Observed flux (ton/yr)')
ax.set_ylabel('log₁₀ Predicted flux (ton/yr)')
ax.set_title(f'Calibrated Meijer Model (LOO-CV): R²={r2_final:.2f}, ρ={sp_final:.2f}')
ax.set_xlim(lims); ax.set_ylim(lims)
ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.savefig(RESULTS / 'calibrated_meijer_scatter.png', dpi=150)
plt.show()

## 5. Build updated ranking with corrected Meijer predictions

Apply the linear calibration to all 31,819 outfalls.

In [ ]:
# Apply calibration to all outfalls
log_me_all = np.log10(fm['emission_ton'].clip(lower=0.01))
fm['log_pred_calibrated'] = lr_final.predict(log_me_all.values.reshape(-1,1))
fm['pred_calibrated_ton_yr'] = 10 ** fm['log_pred_calibrated']

# Ranking
fm['calibrated_rank'] = fm['pred_calibrated_ton_yr'].rank(ascending=False).astype(int)
fm['meijer_rank'] = fm['emission_ton'].rank(ascending=False).astype(int)

# Top 50
top50 = fm.nsmallest(50, 'calibrated_rank')
print("Top 50 calibrated predictions:")
cols = ['calibrated_rank', 'meijer_rank', 'pred_calibrated_ton_yr', 'emission_ton']
print(top50[cols].to_string())

In [ ]:
# How much do ranks change?
rank_delta = (fm['meijer_rank'] - fm['calibrated_rank']).abs()
print(f"Rank changes (all 31,819 outfalls):")
print(f"  Mean |Δrank|: {rank_delta.mean():.1f}")
print(f"  Median |Δrank|: {rank_delta.median():.1f}")
print(f"  Max |Δrank|: {rank_delta.max()}")
print(f"  Top-50 overlap: {len(set(top50.index) & set(fm.nsmallest(50, 'meijer_rank').index))}/50")

# Rank correlation
sp_rank = stats.spearmanr(fm['calibrated_rank'], fm['meijer_rank'])
print(f"  Spearman ρ (all ranks): {sp_rank.statistic:.3f}")

## 6. Value-add analysis: where does updated data change predictions?

The calibration slope < 1 means Meijer **overestimates** high-emission rivers.
This is consistent with the literature (Meijer is known to overpredict top emitters).

Our key value-adds:
1. **Updated waste data** (WaW3.0 from 2024 vs 2010 estimates)
2. **Updated climate data** (ERA5-Land 2015-2025 vs older HydroSHEDS)
3. **Nightlight data** (VIIRS 2021 vs 2012 DMSP-OLS)
4. **Calibration correction** from observed flux

Next steps (notebook 05):
- Reimplement Meijer's P(M)×P(R)×P(O) with updated inputs
- Apply calibration from observed data
- Generate updated global ranking

In [ ]:
# Save results
fm.to_csv(DATA_PROC / 'feature_matrix_v2_calibrated.csv', index=False)

# Save calibration model
import joblib
joblib.dump(lr_final, DATA_PROC / 'meijer_calibration_model.pkl')

print(f"Saved calibrated predictions for {len(fm)} outfalls")
print(f"Saved calibration model")

# Summary statistics
print(f"\n=== Summary ===")
print(f"Calibration: log10(obs) = {lr_final.coef_[0]:.3f} × log10(Meijer) + {lr_final.intercept_:.3f}")
print(f"LOO-CV performance: R²={r2_final:.3f}, ρ={sp_final:.3f}")
print(f"Meijer annual: R²={r2_me:.3f} (monthly: ~0.71)")
print(f"Calibration improves R² from {r2_me:.3f} to {r2_final:.3f}")